In [ ]:
# Lab type: extend
# Course: AI402 — Retrieval & RAG Systems
# Lesson: RAG Observability and Cost
# Task: A working two-stage pipeline is provided with NO instrumentation.
# Extend it with a per-stage trace and a per-query cost model, then use
# your own instruments to answer two questions with data.

# Lab: Instrumenting a RAG Pipeline

**Outputs are cleared.** Run every cell top to bottom.

## Setup: the uninstrumented pipeline

In [ ]:
!pip install sentence-transformers rank-bm25 faiss-cpu numpy pandas --quiet

In [ ]:
import numpy as np

# The Nimbus Analytics product knowledge base: (doc_id, heading_path, text)
CORPUS = [
    ("plans-overview", "Pricing > Plans",
     "Nimbus Analytics offers three subscription plans: Starter, Teams, and "
     "Enterprise. Starter includes 5 seats and community support. Teams includes "
     "50 seats, shared dashboards, and priority email support. Enterprise includes "
     "unlimited seats, priority support, and advanced security features."),
    ("sso-policy", "Pricing > Enterprise plan",
     "Single sign-on (SSO) with SAML 2.0 is available on the Enterprise plan only. "
     "The Teams plan does not include SSO. Enterprise customers can configure SSO "
     "from the admin console under Security settings."),
    ("seat-pricing", "Pricing > Seats",
     "Per-seat pricing: Starter is $12 per seat per month, Teams is $29 per seat "
     "per month, and Enterprise pricing is custom. Annual billing gives a 20 "
     "percent discount on all plans."),
    ("refund-policy", "Billing > Refunds",
     "Customers can request a full refund within 30 days of purchase. To get your "
     "money back after 30 days, contact billing support; partial refunds are "
     "prorated for annual subscriptions."),
    ("error-e4022", "Troubleshooting > Error codes",
     "Error E4022 means the API rate limit was exceeded. The Starter plan allows "
     "100 requests per minute, Teams 1,000, and Enterprise 10,000. Wait 60 seconds "
     "and retry, or upgrade the plan."),
    ("error-e5001", "Troubleshooting > Error codes",
     "Error E5001 indicates an expired API token. Rotate the token from the admin "
     "console under API settings. Tokens expire after 90 days by default."),
    ("api-export", "API > Export",
     "The export endpoint POST /v2/export creates a CSV export of dashboard data. "
     "Exports are limited to 100,000 rows on Teams and 1 million rows on "
     "Enterprise."),
    ("data-retention", "Security > Data retention",
     "Event data is retained for 13 months on all plans. Enterprise customers can "
     "configure custom retention windows up to 5 years from the admin console."),
    ("priority-support", "Support > Tiers",
     "Priority support with a 4-hour response SLA is included in Teams and "
     "Enterprise plans. Starter includes community support only."),
    ("dashboard-sharing", "Product > Dashboards",
     "Shared dashboards let teammates view and edit the same dashboard. Sharing "
     "outside your workspace requires a public link, available on Teams and "
     "Enterprise."),
    ("audit-logs", "Security > Audit logs",
     "Audit logs record sign-ins, permission changes, and data exports. Audit "
     "logs are an Enterprise-only feature and are retained for 2 years."),
    ("cancel-downgrade", "Billing > Cancellation",
     "You can cancel or downgrade at any time from the billing page. Downgrades "
     "take effect at the end of the current billing period."),
]
DOC_IDS = [d[0] for d in CORPUS]
DOC_TEXTS = [f"{d[1]}: {d[2]}" for d in CORPUS]

# Labelled evaluation queries: (query, set of relevant doc_ids)
EVAL_SET = [
    ("does the teams plan include sso", {"sso-policy"}),
    ("how do I get my money back", {"refund-policy"}),
    ("what does error E4022 mean", {"error-e4022"}),
    ("how long is event data kept", {"data-retention"}),
    ("cost per seat on the teams plan", {"seat-pricing"}),
    ("response time for priority support", {"priority-support"}),
    ("row limit for csv export", {"api-export"}),
    ("rotate an expired api token", {"error-e5001"}),
]
print(f"{len(CORPUS)} documents, {len(EVAL_SET)} labelled queries")

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def embed(texts):
    return embedder.encode(list(texts), normalize_embeddings=True)

DOC_EMB = embed(DOC_TEXTS)

def dense_search(query, k=5, doc_emb=None, doc_ids=None):
    doc_emb = DOC_EMB if doc_emb is None else doc_emb
    doc_ids = DOC_IDS if doc_ids is None else doc_ids
    scores = doc_emb @ embed([query])[0]
    order = np.argsort(scores)[::-1][:k]
    return [(doc_ids[i], float(scores[i])) for i in order]

print(dense_search("does the teams plan include sso", k=3))

In [ ]:
from rank_bm25 import BM25Okapi

def tokenize(text):
    return [t.strip('.,:;()$').lower() for t in text.split()]

bm25 = BM25Okapi([tokenize(t) for t in DOC_TEXTS])

def rrf_fuse(rankings, k=60):
    scores = {}
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking, start=1):
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank)
    return sorted(scores, key=scores.get, reverse=True)

INDEX_META = {"index_version": "2026-09-02", "embedding_model":
              "sentence-transformers/all-MiniLM-L6-v2"}

def rag_answer(query, top_k=3):
    dense_ids = [d for d, _ in dense_search(query, k=8)]
    bm25_scores = bm25.get_scores(tokenize(query))
    bm25_ids = [DOC_IDS[i] for i in np.argsort(bm25_scores)[::-1][:8]]
    fused = rrf_fuse([dense_ids, bm25_ids])[:top_k]
    texts = {d: t for d, t in zip(DOC_IDS, DOC_TEXTS)}
    context = "\n\n".join(texts[d] for d in fused)
    prompt = f"Context:\n{context}\n\nQuestion: {query}"
    return prompt  # (generation call omitted — we instrument up to it)

print(rag_answer("does the teams plan include sso")[:200], "...")

## Extension 1: the per-stage trace

Rewrite the pipeline as `rag_answer_traced(query)` returning `(prompt, trace)` where `trace` records: the raw query, each arm's ranked doc IDs, the fused list, the assembled context's doc IDs, the index metadata from `INDEX_META`, and a token estimate for the prompt (`len(prompt) // 4`). Record chunk/doc **IDs**, never full text — why?

In [ ]:
def rag_answer_traced(query, top_k=3):
    # TODO: same pipeline, plus a trace dict per the spec above
    pass


<details>
<summary>🔑 Reveal model answer — Extension 1</summary>

```python
def rag_answer_traced(query, top_k=3):
    dense_ids = [d for d, _ in dense_search(query, k=8)]
    bm25_scores = bm25.get_scores(tokenize(query))
    bm25_ids = [DOC_IDS[i] for i in np.argsort(bm25_scores)[::-1][:8]]
    fused = rrf_fuse([dense_ids, bm25_ids])[:top_k]
    texts = {d: t for d, t in zip(DOC_IDS, DOC_TEXTS)}
    context = "\n\n".join(texts[d] for d in fused)
    prompt = f"Context:\n{context}\n\nQuestion: {query}"
    trace = {
        "query_raw": query,
        "retrieval": {"dense": dense_ids, "bm25": bm25_ids,
                      **INDEX_META},
        "fused": fused,
        "context_docs": fused,
        "prompt_tokens_est": len(prompt) // 4,
    }
    return prompt, trace
```

IDs, not text: a trace store holding full retrieved content is a second copy of the corpus outside its access controls — log readers, caches and retention schedules don't respect tenant boundaries. IDs keep the trace joinable to content under proper authorisation.

</details>

## Extension 2: attribute a planted failure

The query `"how fast is priority support"` should retrieve `priority-support`. Use *only your trace* (not the corpus) to determine: was it in the dense arm's list? the BM25 arm's? the fused top-3? the context? Write one sentence attributing the failure (or success) to a stage.

In [ ]:
# Work here: run rag_answer_traced on the query and inspect the trace.


<details>
<summary>🔑 Reveal model answer — Extension 2</summary>

```python
_, tr = rag_answer_traced("how fast is priority support")
for stage in ("dense", "bm25"):
    print(stage, "priority-support" in tr["retrieval"][stage],
          tr["retrieval"][stage])
print("fused top-3:", tr["fused"])
```

Typical finding: both arms retrieve `priority-support` and it survives fusion — the stage-by-stage walk that took one dict lookup here is exactly the walk that is *impossible* when only the final prompt is logged. If it had been missing from the fused list but present in an arm, the attribution would be "fusion/ranking", and so on up the funnel.

</details>

## Extension 3: the cost model

Assume: generation input $3.00 per million tokens; query embedding $0.02 per million tokens (~queries are tiny); ignore vector-DB read cost. Using `prompt_tokens_est` averaged over all EVAL_SET queries, compute monthly cost at 1M queries/month for top_k=3 vs top_k=8. Then answer: which single number in this lab is the biggest billing knob?

In [ ]:
# Work here: average prompt_tokens_est over EVAL_SET for top_k=3 and 8,
# then monthly_cost = avg_tokens / 1e6 * 3.00 * 1_000_000 queries.


<details>
<summary>🔑 Reveal model answer — Extension 3</summary>

```python
for k in (3, 8):
    toks = [rag_answer_traced(q, top_k=k)[1]["prompt_tokens_est"]
            for q, _ in EVAL_SET]
    avg = sum(toks) / len(toks)
    monthly = avg / 1e6 * 3.00 * 1_000_000
    print(f"top_k={k}: avg {avg:.0f} input tokens/query -> "
          f"${monthly:,.0f}/month at 1M queries")
```

The retrieved-context size — `top_k` × chunk length — is the dominant knob: every retrieved chunk is billed again as generation input on every query, and it scales linearly with both. This is why "just retrieve more" has a precise monthly price, and why prompt caching can't rescue it (retrieved context differs per query).

</details>

## Summary

1. Traces record chunk _______, never full text.
2. Failure attribution = checking each stage's recorded output for the expected doc, from _______ backwards.
3. The dominant RAG cost line is retrieved context billed as _______.

<details>
<summary>🔑 Reveal summary answers</summary>

1. **IDs**
2. **generation/context assembly** (the last stage)
3. **generation input tokens**

</details>